# Part 6b: Symbolic Regression

In this notebook we will train a **symbolic regression (SR)** model on the LHC jet tagging dataset and convert it to a low-latency, low-resource FPGA design with hls4ml.

## What is Symbolic Regression?

Symbolic regression is a machine learning technique that searches for a **mathematical expression** — a formula containing elementary operations (+, -, ×) and functions (sin, cos etc.) — that best fits the training data; for example:

$$f(x) = \sin(x_3 + 0.5 \cdot x_{14}) \cdot (x_2 - 1.2)$$

### How is it trained?

SR is typically solved with **genetic programming**: a population of candidate expressions evolves over many generations. At each step, expressions are mutated (e.g. a node is swapped for a different operator) and crossed over (subtrees are exchanged between two parent expressions). Candidate functions are selected for survival based on a fitness function that penalises both prediction error and expression complexity, driving the search towards simple, accurate formulas.

### Why use SR for FPGA inference?

Neural networks rely on repeated **multiply-accumulate (MAC)** operations arranged across many layers. This translates directly to a large number of DSP blocks and LUT resources on the FPGA, and the depth of the network drives up latency.

Symbolic regression takes a different trade-off: it produces fewer but more complex operations — `sin`, `cos`, arithmetic — applied in a compact formula. Because there are far fewer operations overall, both latency and resource usage can drop substantially compared to a neural network of equivalent accuracy.

These complex mathematical functions can also be efficiently approximated with **lookup tables**, which can further reduce the resource consumption, as shown in the rest of this notebook.

One important caveat: SR search is combinatorial and scales poorly with the number of input features, so it is best suited to problems with a small-to-moderate input dimensionality, such as the 16-feature jet tagging task here.

### PySR

[PySR](https://github.com/MilesCranmer/PySR) is a high-performance Python library for symbolic regression based on genetic programming. Its search backend is written in Julia, which it manages automatically: on the first run inside a fresh environment, PySR will download and install Julia — **this can take 5–10 minutes**. Subsequent runs start immediately.

## Key notebook parts

- **Model training**: run PySR to find symbolic expressions for each of the five jet classes
- **hls4ml conversion**: convert the expressions into a synthesisable HLS C++ project, in two flavours:
  - **Standard** (`hls_model`): uses the Vivado/Vitis HLS math library for `sin`/`cos`
  - **LUT-approximated** (`hls_model_lut`): replaces `sin`/`cos` with lookup-table approximations to save FPGA resources
- **Performance comparison**: compare accuracy and ROC curves between the two HLS implementations
- **Synthesis**: run Vivado/Vitis HLS C-synthesis to estimate latency and resource usage

In [ ]:
import os
import numpy as np
import sympy
import matplotlib.pyplot as plt
import hls4ml
from scipy.special import softmax
from sklearn.metrics import roc_curve, auc, accuracy_score
from pysr import PySRRegressor

%matplotlib inline

## Load the jet tagging dataset

We load the preprocessed arrays saved by `1_getting_started/1a_train_keras.ipynb` or `1b_train_pytorch.ipynb`. Run either notebook first.

PySR uses `L2MarginLoss`, a margin-based loss that expects class labels in $\{-1, +1\}$ rather than the one-hot $\{0, 1\}$ encoding used in earlier parts. We convert with $Y = 2Y_{\text{one-hot}} - 1$.

In [ ]:
X_train_val = np.load('../data/jet-tagging/X_train_val.npy')
X_test      = np.load('../data/jet-tagging/X_test.npy')
y_train_val = np.load('../data/jet-tagging/y_train_val.npy')
y_test      = np.load('../data/jet-tagging/y_test.npy')
classes     = np.load('../data/jet-tagging/classes.npy', allow_pickle=True)

# Convert one-hot {0,1} → margin labels {-1,+1} required by L2MarginLoss
Y_train_val = 2 * y_train_val - 1
Y_test      = 2 * y_test      - 1

print(f'Training set: {X_train_val.shape}, Test set: {X_test.shape}')

## Limit the training set

Genetic programming explores a combinatorially large space of expressions and evaluates each candidate on the full training set at every generation. Runtime therefore scales roughly linearly with the number of training samples. Using the full 580k-sample dataset would make each generation prohibitively slow. We limit to 8,000 samples — enough to guide the search towards good expressions while keeping the training fast.

In [ ]:
N_TRAIN = 8000
X_train = X_train_val[:N_TRAIN]
Y_train = Y_train_val[:N_TRAIN]

print(f'X_train: {X_train.shape},  Y_train: {Y_train.shape}')
print(f'X_test:  {X_test.shape},   Y_test:  {Y_test.shape}')

## Train SR with PySR

We configure PySR with a restricted operator set to keep expressions compact and FPGA-friendly:

- **`binary_operators`**: `+`, `-`, `*` — the basic arithmetic building blocks
- **`unary_operators`**: `sin` and a custom `sc(x) = sin(x)·cos(x)` shorthand
- **`constraints`** / **`nested_constraints`**: prevent deeply nested trigonometric calls. Without this restriction, expressions like `sin(sin(sc(x + 1.3)))` could appear — deeply nested functions inflate the number of hardware operations and increase latency without a proportional gain in accuracy.
- **`select_k_features=6`**: PySR internally selects the 6 most informative features out of the 16 available, reducing the search space
- **`loss='L2MarginLoss()'`**: $(1 - y \cdot \hat{y})^2$, a margin loss on the raw (pre-softmax) output
- **`timeout_in_seconds=600`**: the search stops after 10 minutes regardless of iteration count

**Note:** if this is the first time running PySR in this environment, it will download and install Julia before starting the search. This is a one-time setup that takes approximately 5–10 minutes and is separate from the 10-minute training time.

In [ ]:
model_pysr = PySRRegressor(
    model_selection='accuracy',
    niterations=20,
    timeout_in_seconds=600,
    maxsize=30,
    select_k_features=6,
    binary_operators=['+', '-', '*'],
    unary_operators=['sin', 'sc(x)=sin(x)*cos(x)'],
    complexity_of_operators={'+': 1, '-': 1, '*': 1, 'sin': 1, 'sc': 1},
    constraints={'sin': 20, 'sc': 20},
    nested_constraints={'sin': {'sin': 0, 'sc': 0}, 'sc': {'sin': 0, 'sc': 0}},
    extra_sympy_mappings={'sc': lambda x: sympy.sin(x) * sympy.cos(x)},
    loss='L2MarginLoss()',
)
model_pysr.fit(X_train, Y_train)

**Note:** With the settings above the model typically achieves around **70% classification accuracy** on the test set. Training is kept deliberately short so the tutorial runs in a reasonable time — increasing `timeout_in_seconds` or `niterations` will allow PySR to explore more of the expression space and find better-performing formulas.

## Extract expressions

We extract the best expression found for each of the five jet classes and print them. Two string variants are prepared:
- **`expr`**: uses `sin`/`cos` directly — hls4ml calls the Vivado/Vitis HLS math library
- **`expr_lut`**: replaces them with `sin_lut`/`cos_lut` — hls4ml generates lookup-table approximations

In [ ]:
for i in range(5):
    print(f'Tagger {i} ({classes[i]}) = {model_pysr.sympy()[i]}')
    print('-' * 60)

expr     = [str(model_pysr.sympy()[i]) for i in range(5)]
expr_lut = [e.replace('sin', 'sin_lut').replace('cos', 'cos_lut') for e in expr]

## Set up lookup-table approximations

On FPGAs, computing `sin(x)` or `cos(x)` exactly requires a multi-cycle IP core that can consume significant DSP and LUT resources. An alternative is to pre-compute a table of values and approximate the function by looking up the nearest entry — a **lookup table (LUT)** (but, not to be confused with look-up tables as hardware primitives on AMD/Xilinx FPGAs).

hls4ml's `init_pysr_lut_functions` registers custom function names (`sin_lut`, `cos_lut`) and associates each with a table specification:

- **`range_start` / `range_end`**: the interval over which the table is populated. Values outside this range saturate to the nearest endpoint. The range `[-8, 8]` covers roughly 1.3 full periods of sin/cos and matches the typical argument range in these expressions.
- **`N`**: the number of table entries. With `N=256`, the interval `[-8, 8]` is divided into 256 equal steps of width $16/256 = 0.0625$, giving a maximum approximation error of about $\cos(0.03) - 1 \approx 5 \times 10^{-4}$.

Increasing `N` improves accuracy but increases BRAM usage; narrowing the range also improves accuracy if you know the arguments will be small.

In [ ]:
from hls4ml.utils.symbolic_utils import init_pysr_lut_functions

function_definitions = [
    'sin_lut(x) = math_lut(sin, x, N=256, range_start=-8, range_end=8)',
    'cos_lut(x) = math_lut(cos, x, N=256, range_start=-8, range_end=8)',
]
init_pysr_lut_functions(init_defaults=True, function_definitions=function_definitions)

lut_functions = {
    'sin_lut': {'math_func': 'sin', 'range_start': -8, 'range_end': 8, 'table_size': 256},
    'cos_lut': {'math_func': 'cos', 'range_start': -8, 'range_end': 8, 'table_size': 256},
}

## Parse expressions to sympy

hls4ml expects sympy expression objects. We parse both string lists back into sympy format.

In [ ]:
for i in range(len(expr)):
    expr[i]     = sympy.parsing.sympy_parser.parse_expr(expr[i])
    expr_lut[i] = sympy.parsing.sympy_parser.parse_expr(expr_lut[i])

## Convert to hls4ml

`convert_from_symbolic_expression` takes the list of sympy expressions (one per output class) and generates a Vivado/Vitis HLS C++ project. `n_symbols=16` tells hls4ml that the input has 16 features (`x0`–`x15`), matching the jet tagging dataset.

**Vivado/Vitis HLS paths:** if `hls_model.compile()` raises an error about missing header files or shared libraries, you need to point hls4ml at your local Vivado/Vitis HLS installation. Uncomment and adjust the `HLS_INCLUDE_PATH` / `HLS_LIBS_PATH` lines in the cell below and pass them as arguments. The paths will differ depending on your Vivado/Vitis HLS version and installation location.

In [ ]:
# Uncomment and adjust these paths if hls_model.compile() raises errors
# about missing header files or shared libraries.
# HLS_INCLUDE_PATH = '/tools/Xilinx/Vitis_HLS/2024.1/include'
# HLS_LIBS_PATH = '/tools/Xilinx/Vitis_HLS/2024.1/lnx64'

hls_model = hls4ml.converters.convert_from_symbolic_expression(
    expr,
    n_symbols=16,
    output_dir='../hls4ml_prjs/hls4ml_prj_sr_part6b',
    precision='ap_fixed<16,6>',
    part='xcu250-figd2104-2L-e',
    # hls_include_path=HLS_INCLUDE_PATH,
    # hls_libs_path=HLS_LIBS_PATH,
)
hls_model.compile()

hls_model_lut = hls4ml.converters.convert_from_symbolic_expression(
    expr_lut,
    n_symbols=16,
    output_dir='../hls4ml_prjs/hls4ml_prj_sr_lut_part6b',
    precision='ap_fixed<16,6>',
    part='xcu250-figd2104-2L-e',
    # hls_include_path=HLS_INCLUDE_PATH,
    # hls_libs_path=HLS_LIBS_PATH,
    lut_functions=lut_functions,
)
hls_model_lut.compile()

## Compare performance on the test set

We run both HLS models over the full test set and compare accuracy and ROC-AUC against the PySR expressions evaluated in floating point.

In [ ]:
Y_hls     = softmax(hls_model.predict(np.ascontiguousarray(X_test)), axis=1)
Y_hls_lut = softmax(hls_model_lut.predict(np.ascontiguousarray(X_test)), axis=1)

y_true = np.argmax(y_test, axis=1)

print(f'HLS   accuracy: {accuracy_score(y_true, np.argmax(Y_hls,     axis=1)):.4f}')
print(f'HLS LUT accuracy: {accuracy_score(y_true, np.argmax(Y_hls_lut, axis=1)):.4f}')

In [ ]:
color = ['blue', 'orange', 'green', 'red', 'purple']
fig, ax = plt.subplots(figsize=(10, 8))

for Y_pred, label, ls in [(Y_hls, 'HLS', '-'), (Y_hls_lut, 'HLS LUT', '--')]:
    for c, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test[:, c], Y_pred[:, c])
        ax.plot(tpr, fpr, ls=ls, color=color[c],
                label=f'{cls}, AUC={auc(fpr, tpr):.2f}' if label == 'HLS' else '_nolegend_',
                lw=1.5)

from matplotlib.lines import Line2D
from matplotlib.legend import Legend
style_lines = [Line2D([0], [0], ls=ls, color='k') for ls in ['-', '--']]
leg1 = ax.legend(loc='lower right', fontsize=10, frameon=False)
leg2 = Legend(ax, style_lines, ['HLS', 'HLS LUT'], loc='center right', frameon=False)
ax.add_artist(leg1)
ax.add_artist(leg2)

ax.set_yscale('log')
ax.set_xlabel('True positive rate', size=13)
ax.set_ylabel('False positive rate', size=13)
ax.set_xlim(0, 1)
ax.set_ylim(0.001, 1)
ax.grid(True)

## Synthesize

Run Vivado/Vitis HLS C-synthesis to get latency and resource estimates.

**This can take several minutes.**

In [ ]:
hls_model.build(csim=False)
hls4ml.report.read_vivado_report('../hls4ml_prjs/hls4ml_prj_sr_part6b')

In [ ]:
hls_model_lut.build(csim=False)
hls4ml.report.read_vivado_report('../hls4ml_prjs/hls4ml_prj_sr_lut_part6b')